In [ ]:
import pandas as pd
import numpy as np
import re
import eurostat 

In [ ]:
# Display options so DataFrame output isn't truncated with "..."
pd.set_option("display.max_rows", None)      # show all rows (use a number e.g. 200 if it gets too long)
pd.set_option("display.max_columns", None)   # show all columns
pd.set_option("display.width", None)         # don't wrap based on terminal width
pd.set_option("display.max_colwidth", None)  # don't truncate cell contents

In [ ]:
EU_COUNTRIES = [
    'BE', 'BG', 'CZ', 'DK', 'DE', 'EE', 'IE', 
    'EL', 'ES', 'FR', 'HR', 'IT', 'CY', 'LV',
    'LT', 'LU', 'HU', 'MT', 'NL', 'AT', 'PL', 
    'PT', 'RO', 'SI', 'SK', 'FI', 'SE'
]

EFTA_COUNTRIES = ['IS', 'LI', 'NO', 'CH']

EU_EFTA = EU_COUNTRIES + EFTA_COUNTRIES

In [ ]:
def nace_section_or_nan(s: str) -> str | float:
    s = str(s).strip().upper()
    match = re.fullmatch(r'([A-U])', s)

    if match:
        return match.group(1)
    else:
        return np.nan

## AI adoption (2021-2025 with gap year 2022)

In [ ]:
ain2 = eurostat.get_data_df('isoc_eb_ain2')
print(ain2)

In [ ]:
ain2.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)
ain2.describe()

In [ ]:
ain2.describe(include=["object", "bool"])

In [ ]:
# Filter the data 
# E_AI_TANY - Enterprises use at least one of the AI technologies

ai_adopt = ain2.copy()

ai_adopt = ai_adopt[
    (ai_adopt["indic_is"] == "E_AI_TANY") &
    (ai_adopt["unit"] == "PC_ENT") & 
    (ai_adopt["size_emp"] == "GE10")
]

ai_adopt = ai_adopt.drop(columns=['indic_is','unit',  'freq', 'size_emp'])

ai_adopt = ai_adopt[ai_adopt['geo'].isin(EU_EFTA)]
print(ai_adopt)

In [ ]:
check_S = ai_adopt[ai_adopt["nace_r2"] == "S"].count()
print(check_S)

In [ ]:
ai_adopt['nace_r2_1d'] = ai_adopt['nace_r2'].map(nace_section_or_nan)
print(ai_adopt)

In [ ]:
ai_adopt = ai_adopt.dropna()
print(pd.unique(ai_adopt['nace_r2']))
print(ai_adopt)

In [ ]:
# Create the 2022 column by averaging 2021 and 2023
ai_adopt['2022'] = (ai_adopt['2021'] + ai_adopt['2023']) / 2

# Check the results
print(ai_adopt.head())

In [ ]:
ai_adopt = ai_adopt.drop(columns='nace_r2_1d')
cols_to_melt = ['2021', '2022', '2023', '2024', '2025']

df_ai_panel = ai_adopt.melt(
    id_vars=['geo', 'nace_r2'],      
    value_vars=cols_to_melt,        
    var_name='year_raw',             
    value_name='ai_adoption'        
)


df_ai_panel['year'] = df_ai_panel['year_raw'].str.extract(r'(\d+)').astype(int)
df_ai_panel.drop(columns=['year_raw'], inplace=True)
df_ai_panel = df_ai_panel.sort_values(by=['geo', 'nace_r2', 'year']).reset_index(drop=True)
print(df_ai_panel.head(10))

In [ ]:
df_ai_panel.to_csv('data_panel/ai_adopt.csv', index = False)

## ICT training (2020, 2022, 2024)

In [ ]:
train = eurostat.get_data_df('isoc_ske_ittn2')
print(train)


In [ ]:
train.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)
train.drop(['2012', '2014', '2015', '2016', '2017', '2018', '2019'], axis='columns', inplace=True)
train.info()

In [ ]:
# Filter the data 
train_new = train.copy()
train_new = train_new[
    (train_new['indic_is'] == 'E_ITT2') & # Enterprise provided training to their personnel to develop their ICT skills
    (train_new['unit'] == 'PC_ENT') & # Percentage of enterprises 
    (train_new['size_emp'] == 'GE10')
]

train_new = train_new.drop(columns=['freq', 'size_emp', 'indic_is', 'unit'])
train_new = train_new[train_new['geo'].isin(EU_EFTA)]

In [ ]:
train_new['nace_r2_1d'] = train_new['nace_r2'].map(nace_section_or_nan)
train_new.drop(columns=['nace_r2'], inplace=True) 
train_new.rename(columns={'nace_r2_1d' : 'nace_r2'}, inplace=True)
train_new = train_new.dropna(subset=['nace_r2'])
print(train_new)

In [ ]:
from sklearn.impute import KNNImputer

year_cols = ['2020', '2022', '2024']
missing_masks = {col: train_new[col].isna() for col in year_cols}


def impute_by_sector(group):
    n_rows = len(group)
    if n_rows > 1:
        neighbors = min(5, n_rows - 1)
        sector_imputer = KNNImputer(n_neighbors=neighbors, weights='distance')
        
        valid_cols = group[year_cols].dropna(axis=1, how='all').columns
        if len(valid_cols) > 0:
            group[valid_cols] = sector_imputer.fit_transform(group[valid_cols])
            
    return group

train_new = train_new.groupby('nace_r2', group_keys=False).apply(impute_by_sector)

# impute the 2020 year using macroeconomic deflation ratios for remaining missing values
country_means = train_new.groupby('geo')[['2020', '2022']].mean()

# Calculate the deflation ratio (Country Avg 2020 / Country Avg 2022)
country_means['deflation_ratio'] = np.where(
    country_means['2022'] != 0, 
    country_means['2020'] / country_means['2022'], 
    np.nan
)

ratio_map = country_means['deflation_ratio'].to_dict()
train_new['macro_ratio'] = train_new['geo'].map(ratio_map)

# Apply the formula ONLY where 2020 is STILL missing and 2022 exists
mask = train_new['2020'].isna() & train_new['2022'].notna()
train_new.loc[mask, '2020'] = train_new.loc[mask, '2022'] * train_new.loc[mask, 'macro_ratio']
train_new = train_new.drop(columns=['macro_ratio'])

for col in year_cols:
    train_new[f'is_{col}_imputed'] = (missing_masks[col] & train_new[col].notna()).astype(int)

In [ ]:
print(train_new)

In [ ]:
cols_to_melt = ['2020', '2022', '2024']

df_train_panel = train_new.melt(
    id_vars=['geo', 'nace_r2'],      
    value_vars=cols_to_melt,         
    var_name='year_raw',            
    value_name='training_ict'        
)

df_train_panel['year'] = df_train_panel['year_raw'].str.extract(r'(\d+)').astype(int)
df_train_panel.drop(columns=['year_raw'], inplace=True)
df_train_panel['training_ict'] = df_train_panel['training_ict'].round(2)
df_train_panel = df_train_panel.sort_values(by=['geo', 'nace_r2', 'year']).reset_index(drop=True)

print(df_train_panel.head(10))

In [ ]:
df_train_panel.to_csv('data_panel/train.csv', index = False)

## ICT specialists (2020, 2022, 2024)

In [ ]:
ict_spec = eurostat.get_data_df('isoc_ske_itspen2')
ict_spec.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)
print(ict_spec)


In [ ]:
ict_spec.drop(['2012', '2014', '2015', '2016', '2017', '2018', '2019'], axis='columns', inplace=True)
ict_spec.info()

In [ ]:
# Filter the data 
# E_ITSP2 - Enterprise employed ICT/IT specialists (reduced comparability with 2007)

ict_spec_new = ict_spec.copy()
ict_spec_new = ict_spec_new[
    (ict_spec_new['unit'] == 'PC_ENT') & # Percentage of enterprises 
    (ict_spec_new['size_emp'] == 'GE10')
]

ict_spec_new = ict_spec_new.drop(columns=['freq', 'size_emp', 'indic_is', 'unit'])
ict_spec_new = ict_spec_new[ict_spec_new['geo'].isin(EU_EFTA)]

ict_spec_new['nace_r2_1d'] = ict_spec_new['nace_r2'].map(nace_section_or_nan)

ict_spec_new.drop(columns=['nace_r2'], inplace=True) 
ict_spec_new.rename(columns={'nace_r2_1d' : 'nace_r2'}, inplace=True)
ict_spec_new = ict_spec_new.dropna(subset=['nace_r2'])
print(ict_spec_new)

In [ ]:
year_cols = ['2020', '2022', '2024']
missing_masks = {col: ict_spec_new[col].isna() for col in year_cols}

def impute_by_sector(group):
    n_rows = len(group)
    if n_rows > 1:
        neighbors = min(5, n_rows - 1)
        sector_imputer = KNNImputer(n_neighbors=neighbors, weights='distance')
        
        valid_cols = group[year_cols].dropna(axis=1, how='all').columns
        if len(valid_cols) > 0:
            group[valid_cols] = sector_imputer.fit_transform(group[valid_cols])
            
    return group

ict_spec_new = ict_spec_new.groupby('nace_r2', group_keys=False).apply(impute_by_sector)

# impute the 2020 year using macroeconomic deflation ratios for remaining missing values
country_means = ict_spec_new.groupby('geo')[['2020', '2022']].mean()

# Calculate the deflation ratio (Country Avg 2020 / Country Avg 2022)
country_means['deflation_ratio'] = np.where(
    country_means['2022'] != 0, 
    country_means['2020'] / country_means['2022'], 
    np.nan
)

ratio_map = country_means['deflation_ratio'].to_dict()
ict_spec_new['macro_ratio'] = ict_spec_new['geo'].map(ratio_map)

# Apply the formula ONLY where 2020 is STILL missing and 2022 exists
mask = ict_spec_new['2020'].isna() & ict_spec_new['2022'].notna()
ict_spec_new.loc[mask, '2020'] = ict_spec_new.loc[mask, '2022'] * ict_spec_new.loc[mask, 'macro_ratio']
ict_spec_new = ict_spec_new.drop(columns=['macro_ratio'])

for col in year_cols:
    ict_spec_new[f'is_{col}_imputed'] = (missing_masks[col] & ict_spec_new[col].notna()).astype(int)

In [ ]:
print(ict_spec_new)

In [ ]:
cols_to_melt = ['2020','2022', '2024']

df_ict_panel = ict_spec_new.melt(
    id_vars=['geo', 'nace_r2'],     
    value_vars=cols_to_melt,       
    var_name='year_raw',            
    value_name='spec_ict'           
)

df_ict_panel['year'] = df_ict_panel['year_raw'].str.extract(r'(\d+)').astype(int)
df_ict_panel.drop(columns=['year_raw'], inplace=True)
df_ict_panel['spec_ict'] = df_ict_panel['spec_ict'].round(2)
df_ict_panel = df_ict_panel.sort_values(by=['geo', 'nace_r2', 'year']).reset_index(drop=True)
print(df_ict_panel.head(10))

In [ ]:
df_ict_panel.to_csv('data_panel/ict_spec.csv', index = False)

## Wages 

### Nominal wage (wage per hour in EUR)

In [ ]:
lc = eurostat.get_data_df('lc_lci_lev')
lc.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)
lc.info()

In [ ]:
lc.drop(['2008', '2012','2016'], axis='columns', inplace=True)

# Filter the data 
# per employee in full-time equivalents, per hour

wg = lc.copy()
wg = wg[
    (wg['unit'] == 'EUR') &
    (wg['lcstruct'] == 'D11') # Wages and Salaries (total)
]

wg = wg.drop(columns=['freq', 'unit', 'lcstruct'])
wg = wg[wg['geo'].isin(EU_EFTA)]

print(wg)

In [ ]:
wg['nace_r2_1d'] = wg['nace_r2'].map(nace_section_or_nan)

wg.drop(columns=['nace_r2'], inplace=True) 
wg.rename(columns={'nace_r2_1d' : 'nace_r2'}, inplace=True)

wg = wg.rename(columns={'2020': 'wg_n_2020',
                        '2021': 'wg_n_2021',
                        '2022' : 'wg_n_2022',
                        '2023' : 'wg_n_2023',
                        '2024' : 'wg_n_2024',
                        '2025' : 'wg_n_2025'})
print(wg)

### HICP (relative to 2015=100)

In [ ]:
hicp = eurostat.get_data_df('prc_hicp_aind')
print(hicp)

In [ ]:
hicp.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)
years_to_drop = [str(year) for year in range(1996, 2020)]
hicp.drop(columns=years_to_drop, errors='ignore', inplace=True)
hicp.info()

In [ ]:
# Filter the data 
# annual average index

hicp_inx = hicp.copy()
hicp_inx = hicp_inx.reset_index(drop=True)
hicp_inx = hicp_inx[
    (hicp_inx['unit'] == 'INX_A_AVG') &
    (hicp_inx['coicop'] == 'CP00') 
]

hicp_inx = hicp_inx.drop(columns=['freq', 'unit', 'coicop'])
hicp_inx = hicp_inx[hicp_inx['geo'].isin(EU_EFTA)]

print(hicp_inx)

In [ ]:
cols_to_melt = ['2020', '2021', '2022', '2023', '2024', '2025']

hicp_panel = hicp_inx.melt(
    id_vars=['geo'],     
    value_vars=cols_to_melt,       
    var_name='year_raw',            
    value_name='hicp'           
)

hicp_panel['year'] = hicp_panel['year_raw'].str.extract(r'(\d+)').astype(int)
hicp_panel.drop(columns=['year_raw'], inplace=True)
hicp_panel = hicp_panel.sort_values(by=['geo', 'year']).reset_index(drop=True)
print(hicp_panel.head(10))

In [ ]:
hicp_panel.to_csv('data_panel/hicp.csv', index = False)

In [ ]:
hicp_inx = hicp_inx.rename(columns={'2020': 'hicp_inx_2020',
                        '2021': 'hicp_inx_2021',
                        '2022' : 'hicp_inx_2022',
                        '2023' : 'hicp_inx_2023',
                        '2024' : 'hicp_inx_2024', 
                        '2025' : 'hicp_inx_2025'})

### Calculate real wages (2021-2025)

In [ ]:
wage_countries = set(wg['geo'].unique())
hicp_countries = set(hicp_inx['geo'].unique())

missing_countries = wage_countries - hicp_countries

if len(missing_countries) > 0:
    print(f"WARNING: The following countries are in the Wage data but MISSING in HICP data:\n{missing_countries}")
else:
    print("All countries in Wage data have a matching HICP record.")

In [ ]:
wg_merged = pd.merge(wg, hicp_inx, on='geo', how='left')

In [ ]:
years = ['2020', '2021', '2022', '2023', '2024', '2025']

for year in years:
    # Define column names
    nom_col = f'wg_n_{year}'  
    hicp_col = f'hicp_inx_{year}'  
    real_col = f'wg_r_{year}'  
    
    # Apply formula
    wg_merged[real_col] = (wg_merged[nom_col] / wg_merged[hicp_col]) * 100

print(wg_merged)

In [ ]:
wg_merged = wg_merged.drop(['wg_n_2020', 'wg_n_2021', 'wg_n_2022', 'wg_n_2023', 'wg_n_2024', 'wg_n_2025',
                            'hicp_inx_2020', 'hicp_inx_2021', 'hicp_inx_2022', 'hicp_inx_2023', 'hicp_inx_2024', 'hicp_inx_2025'], axis='columns')
wg_merged = wg_merged.dropna()
print(wg_merged)

In [ ]:
wg_panel = wg_merged.melt(
    id_vars=['geo', 'nace_r2'],    
    value_vars=['wg_r_2020', 'wg_r_2021', 'wg_r_2022', 'wg_r_2023', 'wg_r_2024', 'wg_r_2025'], 
    var_name='year_raw',            
    value_name='real_wage'          
)

wg_panel['year'] = wg_panel['year_raw'].str.extract(r'(\d+)').astype(int)
wg_panel.drop(columns=['year_raw'], inplace=True)
wg_panel = wg_panel.sort_values(by=['geo', 'nace_r2', 'year']).reset_index(drop=True)

print("Panel Format Ready:")
print(wg_panel.head(10))

In [ ]:
wg_panel.to_csv('data_panel/wage.csv', index = False)

## Productivity (2021-2025)

### GVA

In [ ]:
gva = eurostat.get_data_df('nama_10_a64')

In [ ]:
gva.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)
gva.info()

In [ ]:
years_to_drop = [str(year) for year in range(1975, 2021)]
gva = gva.drop(columns=years_to_drop, errors='ignore')
gva['nace_r2_1d'] = gva['nace_r2'].map(nace_section_or_nan)
gva.drop(columns=['nace_r2'], inplace=True) 
gva.rename(columns={'nace_r2_1d' : 'nace_r2'}, inplace=True)
gva = gva[gva['geo'].isin(EU_EFTA)]

In [ ]:
gva = gva[
    (gva['na_item'] == 'B1G') &         # Gross value added
    (gva['unit'] == 'CP_MEUR')        # Current prices, million EUR
]
# print(gva)

### Employees 

In [ ]:
emp_df = eurostat.get_data_df('nama_10_a64_e')

emp = emp_df[
    (emp_df['na_item'] == 'EMP_DC') &     # Employment, domestic concept
    (emp_df['unit'] == 'THS_PER')       # Thousands of persons
]
print(emp)

In [ ]:
emp.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)
years_to_drop = [str(year) for year in range(1975, 2021)]
emp = emp.drop(columns=years_to_drop, errors='ignore')
emp['nace_r2_1d'] = emp['nace_r2'].map(nace_section_or_nan)
emp.drop(columns=['nace_r2'], inplace=True) 
emp.rename(columns={'nace_r2_1d' : 'nace_r2'}, inplace=True)
emp = emp[emp['geo'].isin(EU_EFTA)]
print(emp)

### Productivity (calc) - keep 2021 and 2024 for now

In [ ]:
gva = gva[['geo', 'nace_r2', '2021', '2022', '2023', '2024', '2025']].copy()
emp = emp[['geo', 'nace_r2', '2021', '2022', '2023', '2024', '2025']].copy()

gva = gva.rename(columns={str(y): f'gva_{y}' for y in [2021, 2022, 2023, 2024, 2025]})
emp = emp.rename(columns={str(y): f'emp_{y}' for y in [2021, 2022, 2023, 2024, 2025]})

prod = gva.merge(emp, on=['geo', 'nace_r2'], how='inner')

for y in [2021, 2022, 2023, 2024, 2025]:
    prod[f'gva_{y}'] = pd.to_numeric(prod[f'gva_{y}'], errors='coerce')
    prod[f'emp_{y}'] = pd.to_numeric(prod[f'emp_{y}'], errors='coerce')

print(prod)

In [ ]:
# productivity (EUR per employee) for each year
for y in [2021, 2022, 2023, 2024, 2025]:
    gva_col = f'gva_{y}'    # million EUR
    emp_col = f'emp_{y}'    # thousand persons
    prod_col = f'prod_{y}'  # EUR per employee

    prod[prod_col] = (
        prod[gva_col]     # in thousand EUR
    ) / (
        prod[emp_col]          # persons
    )

    # Clean impossible values
    prod[prod_col] = prod[prod_col].replace([np.inf, -np.inf], np.nan)


print(prod.head())
print("\nCoverage:")
print("Countries:", prod['geo'].nunique())
print("NACE sectors:", prod['nace_r2'].unique())

In [ ]:
prod_long = prod.melt(
    id_vars=['geo', 'nace_r2'],
    value_vars=[f'prod_{y}' for y in [2021, 2022, 2023, 2024, 2025]],
    var_name='year',
    value_name='productivity'
)
prod_long['year'] = prod_long['year'].str.replace('prod_', '').astype(int)
prod_long = prod_long.dropna(subset=['productivity', 'nace_r2'])
prod_long['productivity'] = round(prod_long['productivity'],2)
print(prod_long.head())

In [ ]:
prod_long.to_csv('data_panel/product.csv', index = False)

## Firm size

In [ ]:
fsi =  eurostat.get_data_df('sbs_sc_ovw') 
print(fsi)

In [ ]:
fsi.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)
fsi['nace_r2_1d'] = fsi['nace_r2'].map(nace_section_or_nan)
fsi.drop(columns=['nace_r2'], inplace=True) 
fsi.rename(columns={'nace_r2_1d' : 'nace_r2'}, inplace=True)
fsi = fsi[fsi['geo'].isin(EU_EFTA)]
fsi.info()

In [ ]:
# FSI Calculation (2021-2024) 

df_emp = fsi[fsi['indic_sbs'] == 'EMP_NR'].copy()
df_fsi = df_emp[df_emp['size_emp'].isin(['GE250', 'TOTAL'])].copy()

# Transform year columns into rows
df_melted = df_fsi.melt(
    id_vars=['geo', 'nace_r2', 'size_emp'],     
    value_vars=['2021', '2022', '2023', '2024'], 
    var_name='year_raw',
    value_name='emp_value'
)

df_melted['year'] = df_melted['year_raw'].str.extract(r'(\d+)').astype(int)

df_pivoted = df_melted.pivot_table(
    index=['geo', 'nace_r2', 'year'], 
    columns='size_emp',
    values='emp_value',
    aggfunc='sum'
).reset_index()

df_pivoted.rename(columns={'GE250': 'Emp_250_plus', 'TOTAL': 'Emp_Total'}, inplace=True)

df_pivoted['FSI'] = (df_pivoted['Emp_250_plus'] / df_pivoted['Emp_Total']) * 100

df_fsi_final = df_pivoted[['geo', 'nace_r2', 'year', 'FSI']].copy()

print("FSI Calculation Head (All Years):")
print(df_fsi_final.head())

In [ ]:
df_fsi_final.to_csv('data_panel/fsi.csv', index = False)

## Education

In [ ]:
educ =  eurostat.get_data_df('edat_lfs_9910') 

In [ ]:
educ.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)
educ.drop(['2008', '2009', '2010', '2011','2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019'], axis='columns', inplace=True)
educ.info()

In [ ]:
# Filter the data 
# unit - PC, percent

educ = educ.copy()
educ = educ[
    (educ['sex'] == 'T')& # for all genders
    (educ['age'] == 'Y18-69')& 
    (educ['isced11'] == 'ED5-8')   # 5-8: Tertiary education (levels 5-8)
]

educ = educ.drop(columns=['freq', 'unit', 'age', 'sex', 'isced11'])
educ = educ[educ['geo'].isin(EU_EFTA)]
educ = educ.dropna()

In [ ]:
education_long = educ.melt(
    id_vars=['nace_r2', 'geo'],
    value_vars=['2020', '2021', '2022', '2023', '2024', '2025'],
    var_name='year',
    value_name='tert_edu'
)

education_long['year'] = education_long['year'].astype(int)

education_long['tert_edu'] = pd.to_numeric(
    education_long['tert_edu'], 
    errors='coerce'
)

education_long = education_long[
    education_long['year'].isin([2021, 2022, 2023, 2024, 2025])
].copy()

print(education_long.head())

In [ ]:
education_long.to_csv('data_panel/educ.csv', index = False)

## High skills

In [ ]:
occup = eurostat.get_data_df('lfsa_eisn2')
# print(occup)

In [ ]:
occup.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)
occup.drop(['2008', '2009', '2010', '2011','2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019'], axis='columns', inplace=True)
occup.info()

In [ ]:
# Filter 
# unit - Ths_per, thousandpersons
occup = occup.copy()
occup = occup[
    (occup['sex'] == 'T')& # for all genders
    (occup['age'] == 'Y20-64') # From 20 to 64 years
]

occup = occup.drop(columns=['freq', 'unit', 'age', 'sex'])
occup = occup[occup['geo'].isin(EU_EFTA)]
# print(occup)

In [ ]:
occup['nace_r2_1d'] = occup['nace_r2'].map(nace_section_or_nan)
occup.drop(columns=['nace_r2'], inplace=True) 
occup.rename(columns={'nace_r2_1d' : 'nace_r2'}, inplace=True)

occup = occup.dropna()
# print(occup)

In [ ]:
high_skill_codes = ['OC1', 'OC2', 'OC3']
total_code = 'TOTAL'
relevant_codes = high_skill_codes + [total_code]

occup_filtered = occup[occup['isco08'].isin(relevant_codes)].copy()

df_melted = occup_filtered.melt(
    id_vars=['geo', 'nace_r2', 'isco08'],          
    value_vars=['2020','2021', '2022', '2023', '2024', '2025'], 
    var_name='year_raw',
    value_name='emp_value'
)

df_melted['year'] = df_melted['year_raw'].str.extract(r'(\d+)').astype(int)

occup_pivoted = df_melted.pivot_table(
    index=['geo', 'nace_r2', 'year'], 
    columns='isco08',
    values='emp_value',
    aggfunc='sum'
).reset_index()


occup_pivoted.rename(columns={
    'OC1': 'OC1_Managers',
    'OC2': 'OC2_Professionals',
    'OC3': 'OC3_Technicians',
    'TOTAL': 'TOTAL_Occupied'
}, inplace=True)


occup_pivoted['total_high_skill'] = (
    occup_pivoted['OC1_Managers'] + 
    occup_pivoted['OC2_Professionals'] + 
    occup_pivoted['OC3_Technicians']
)



occup_pivoted['share_high_skill'] = (
    occup_pivoted['total_high_skill'] / occup_pivoted['TOTAL_Occupied']
) * 100



share_high_skill_df = occup_pivoted[['geo', 'nace_r2', 'year', 'share_high_skill']].copy()

share_high_skill_df = share_high_skill_df.dropna()

print("High Skill Share Calculation Head (Panel):")
print(share_high_skill_df.head())

In [ ]:
share_high_skill_df.to_csv('data_panel/share_high_skill.csv', index = False)

## Data without GDP, Unempl, Infl variables

In [ ]:
# education_long
# df_fsi_final
# prod_long
# wg_panel
# df_ict_panel
# df_train_panel
# df_ai_panel
# share_high_skill_df

In [ ]:
print(df_ai_panel)

In [ ]:
df_train_panel['training_ict'] = round(df_train_panel['training_ict'],2)
print(df_train_panel)

In [ ]:
df_ict_panel['spec_ict'] = round(df_ict_panel['spec_ict'],2)
print(df_ict_panel)

In [ ]:
print(education_long)

In [ ]:
prod_long['log_prod'] = np.round(np.log(prod_long['productivity']), 2)
print(prod_long)

In [ ]:
wg_panel['real_wage'] = round(wg_panel['real_wage'],2)
wg_panel['log_wage'] = np.round(np.log(wg_panel['real_wage']), 2)
print(wg_panel)

In [ ]:
# df_fsi_final['FSI'] = round(df_fsi_final['FSI'],2)
# print(df_fsi_final)

In [ ]:
share_high_skill_df['share_high_skill'] = round(share_high_skill_df['share_high_skill'],2)
print(share_high_skill_df)

In [ ]:
# =====================================================================
# 1. BUILD THE NORMAL DATASET (Dependent & Control Variables for t)
# This includes Wages, AI Adoption, and Education (Years: 2021, 2023, 2025)
# =====================================================================
df_normal = wg_panel.copy()
df_normal = pd.merge(df_normal, df_ai_panel, on=['geo', 'nace_r2', 'year'], how='left')
# df_normal = pd.merge(df_normal, share_high_skill_df, on=['geo', 'nace_r2', 'year'], how='left')
df_normal = pd.merge(df_normal, education_long, on=['geo', 'nace_r2', 'year'], how='left')
df_normal = pd.merge(df_normal, prod_long, on=['geo', 'nace_r2', 'year'], how='left')

# =====================================================================
# 2. BUILD THE LAGGED DATASET (Independent Variables for t-1)
# This includes ICT Specialists and Training (Years: 2020, 2022, 2024)
# =====================================================================
df_lag = df_ict_panel.copy()
df_lag = pd.merge(df_lag, df_train_panel, on=['geo', 'nace_r2', 'year'], how='left')

# =====================================================================
# 3. ALIGN THE TIMELINES
# We add +1 to the lagged year so it matches the normal year during the merge.
# (e.g., 2020 becomes 2021, so it joins with the 2021 wage data)
# =====================================================================
df_lag['year'] = df_lag['year'] + 1

# =====================================================================
# 4. FINAL MERGE
# Merge the lagged data into the normal dataset
# =====================================================================
df_master = pd.merge(df_normal, df_lag, on=['geo', 'nace_r2', 'year'], how='left')

# Drop any rows where 'year' is not in our target periods (if any stray years snuck in)
target_years = [2021, 2023, 2025]
df_master = df_master[df_master['year'].isin(target_years)].reset_index(drop=True)

print("\n--- MASTER PANEL DATASET (Lagged Structure) ---")
print(df_master['year'].value_counts().sort_index()) 
print(df_master.head())

In [ ]:
print(df_master)

In [ ]:
# 1. Create a strict dataset dropping any row that has an NA in any column
df_regression = df_master.dropna()

# 2. Check the final number of usable rows
final_n = len(df_regression)
print(f"Total starting rows: {len(df_master)}")
print(f"Final usable rows for regression: {final_n}")

# 3. See how those usable rows are distributed across your periods
print("\nUsable rows by period:")
print(df_regression['year'].value_counts().sort_index())

In [ ]:
df_regression.to_csv('data_panel/panel_master.csv', index = False )

## Data with macro metrics

### Unemployment rate

In [ ]:
unempl = eurostat.get_data_df('tps00203')
print(unempl)

In [ ]:
unempl.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)
unempl.info()

In [ ]:
unempl.drop(['2014', '2015', '2016', '2017', '2018', '2019'], axis='columns', inplace=True)
print(unempl)

In [ ]:
# Filter 
# Y15-74 default age class
# sex total is default


unempl_r = unempl.copy()
unempl_r = unempl_r[ 
    (unempl_r['unit'] == 'PC_ACT') # Percentage of population in the labour force
]

unempl_r = unempl_r.drop(columns=['freq', 'unit', 'age', 'sex'])
unempl_r = unempl_r[unempl_r['geo'].isin(EU_EFTA)]
print(unempl_r)

In [ ]:
unempl_r = unempl_r.dropna()
print(unempl_r)

In [ ]:
# cols_to_melt = ['2020', '2021', '2022', '2023', '2024', '2025']
cols_to_melt = ['2021', '2023', '2025']

unempl_r_panel = unempl_r.melt(
    id_vars=['geo'],     
    value_vars=cols_to_melt,       
    var_name='year_raw',            
    value_name='unempl_r'           
)

unempl_r_panel['year'] = unempl_r_panel['year_raw'].str.extract(r'(\d+)').astype(int)
unempl_r_panel.drop(columns=['year_raw'], inplace=True)
unempl_r_panel = unempl_r_panel.sort_values(by=['geo', 'year']).reset_index(drop=True)
print(unempl_r_panel.head(10))

In [ ]:
unempl_r_panel.to_csv('data_panel/unempl_r.csv', index = False)

### Inflation rate

In [ ]:
infl = eurostat.get_data_df('tec00118')
infl.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)
infl.drop(['2014', '2015', '2016', '2017', '2018', '2019'], axis='columns', inplace=True)
infl.info()

In [ ]:
# Filter 

infl_r = infl.drop(columns=['freq', 'unit', 'coicop18'])
infl_r = infl_r[infl_r['geo'].isin(EU_EFTA)]
print(infl_r)

In [ ]:
infl_r = infl_r.dropna()
print(infl_r)

In [ ]:
infl_r_panel = infl_r.melt(
    id_vars=['geo'],     
    value_vars=cols_to_melt,       
    var_name='year_raw',            
    value_name='infl_r'           
)

infl_r_panel['year'] = infl_r_panel['year_raw'].str.extract(r'(\d+)').astype(int)
infl_r_panel.drop(columns=['year_raw'], inplace=True)
infl_r_panel = infl_r_panel.sort_values(by=['geo', 'year']).reset_index(drop=True)
print(infl_r_panel.head(10))

In [ ]:
infl_r_panel.to_csv('data_panel/infl_r.csv', index = False)

### GDP per capita

In [ ]:
gdp_pc = eurostat.get_data_df('nama_10_pc')
print(gdp_pc)

In [ ]:
gdp_pc.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)
columns_to_drop = gdp_pc.columns[4:49]
gdp_pc.drop(columns=columns_to_drop, inplace=True)
gdp_pc.info()

In [ ]:
gdp_pc = gdp_pc.dropna()
print(gdp_pc)

In [ ]:
# Filter 

gdp = gdp_pc.copy()
gdp = gdp[
    (gdp['na_item'] == 'B1GQ')& # Gross domestic product at market prices
    (gdp['unit'] == 'CP_EUR_HAB') # Current price, euro per capita
]


gdp = gdp.drop(columns=['freq', 'unit', 'na_item'])
gdp = gdp[gdp['geo'].isin(EU_EFTA)]
print(gdp)

In [ ]:
gdp_panel = gdp.melt(
    id_vars=['geo'],     
    value_vars=cols_to_melt,       
    var_name='year_raw',            
    value_name='gdp'           
)

gdp_panel['year'] = gdp_panel['year_raw'].str.extract(r'(\d+)').astype(int)
gdp_panel.drop(columns=['year_raw'], inplace=True)
gdp_panel = gdp_panel.sort_values(by=['geo', 'year']).reset_index(drop=True)
print(gdp_panel.head(10))

In [ ]:
gdp_panel.to_csv('data_panel/gdp.csv', index = False)

## FULL DATASET

In [ ]:
# unempl_r_panel
# infl_r_panel
# gdp_panel

df_full_master = df_regression.copy()
df_full_master = pd.merge(df_full_master, unempl_r_panel, on=['geo', 'year'], how='left')
df_full_master = pd.merge(df_full_master, infl_r_panel, on=['geo', 'year'], how='left')
df_full_master = pd.merge(df_full_master, gdp_panel, on=['geo', 'year'], how='left')
df_full_master['log_gdp'] = np.round(np.log(df_full_master['gdp']), 2)

print("\n--- MASTER PANEL FULL DATASET (2021-2025) ---")
print(df_full_master['year'].value_counts().sort_index()) # Verifies only 2021, 2023, 2025 exist
print(df_full_master)

In [ ]:
print(df_full_master.isna().values.any())

In [ ]:
df_full_master.to_csv('data_panel/full_panel_master.csv', index = False )